<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/NitroAI/brotherhoodKazakhistanAiOlympiad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from transformers import BertModel, BertTokenizer
from scipy.optimize import linear_sum_assignment

In [2]:
train = pd.read_csv("/content/train_data (2).csv")
test = pd.read_csv("/content/test_data (2).csv")
sample = pd.read_csv("/content/sample_output (2).csv")

In [3]:
train

,pair_id,py_source,cpp_source
0,pair_0000,"def f_gold ( num , divisor ) :\n while ( nu...","using namespace std;\nint f_gold ( int num, in..."
1,pair_0001,"def f_gold ( arr , n ) :\n mp = dict ( )\n ...",using namespace std;\nint f_gold ( int arr [ ]...
2,pair_0002,def f_gold ( s ) :\n length = len ( s )\n ...,using namespace std;\nint f_gold ( string str ...
3,pair_0003,"def f_gold ( arr , n ) :\n arr.sort ( ) ;\n...",using namespace std;\nint f_gold ( int arr [ ]...
4,pair_0004,"import sys\n\ndef f_gold ( arr , n ) :\n re...",using namespace std;\nint f_gold ( int arr [ ]...
...,...,...,...
95,pair_0095,"def f_gold ( a , n ) :\n if n == 1 :\n ...","using namespace std;\nint f_gold ( int a [ ], ..."
96,pair_0096,"def f_gold ( stack1 , stack2 , stack3 , n1 , n...",using namespace std;\nint f_gold ( int stack1 ...
97,pair_0097,def f_gold ( n ) :\n position = 1\n m = ...,using namespace std;\nint f_gold ( int n ) {\n...
98,pair_0098,import math\n\ndef f_gold ( n ) :\n if ( n ...,using namespace std;\nint f_gold ( int n ) {\n...


In [4]:
test

,type,datapointID,source
0,query,py_test_public_0000,"def f_gold ( arr , n ) :\n jumps = [ 0 for ..."
1,query,py_test_public_0001,"def f_gold ( s , K ) :\n n = len ( s )\n ..."
2,query,py_test_public_0002,"def f_gold ( arr1 , arr2 , m , n ) :\n i = ..."
3,query,py_test_public_0003,"def f_gold ( arr , low , high ) :\n max = a..."
4,query,py_test_public_0004,"def f_gold ( arr1 , arr2 , m , n , x ) :\n ..."
...,...,...,...
395,candidate,candidate_0195,using namespace std;\nint f_gold ( int n ) {\n...
396,candidate,candidate_0196,using namespace std;\nbool f_gold ( int arr [ ...
397,candidate,candidate_0197,using namespace std;\nint f_gold ( int dist ) ...
398,candidate,candidate_0198,using namespace std;\nint f_gold ( int arr [ ]...


In [5]:
sample

,subtaskID,datapointID,answer
0,1,py_test_public_0000,candidate_0084;candidate_0159;candidate_0015;c...
1,1,py_test_public_0001,candidate_0167;candidate_0181;candidate_0005;c...
2,1,py_test_public_0002,candidate_0007;candidate_0005;candidate_0193;c...
3,1,py_test_public_0003,candidate_0162;candidate_0005;candidate_0071;c...
4,1,py_test_public_0004,candidate_0117;candidate_0005;candidate_0101;c...
...,...,...,...
195,1,py_test_public_0195,candidate_0037;candidate_0026;candidate_0106;c...
196,1,py_test_public_0196,candidate_0079;candidate_0074;candidate_0076;c...
197,1,py_test_public_0197,candidate_0177;candidate_0181;candidate_0073;c...
198,1,py_test_public_0198,candidate_0108;candidate_0131;candidate_0023;c...


In [9]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.optimize import linear_sum_assignment

device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/content/train_data (2).csv")
test = pd.read_csv("/content/test_data (2).csv")
sample = pd.read_csv("/content/sample_output (2).csv")

PY_COL  = "py_source"
CPP_COL = "cpp_source"

MODEL_NAME = "google-bert/bert-base-uncased"
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 4
LR = 1e-5
TEMPERATURE = 0.02

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDataset(Dataset):
    def __init__(self, df):
        self.py = df[PY_COL].astype(str).tolist()
        self.cpp = df[CPP_COL].astype(str).tolist()
    def __len__(self):
        return len(self.py)
    def __getitem__(self, i):
        return self.py[i], self.cpp[i]

def collate(batch):
    py, cpp = zip(*batch)
    py_tok = tokenizer(list(py), padding=True, truncation=True,
                       max_length=MAX_LEN, return_tensors="pt")
    cpp_tok = tokenizer(list(cpp), padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
    return py_tok, cpp_tok

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

class DualEncoder(nn.Module):
    def __init__(self, share_weights=False):
        super().__init__()
        self.py_encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.cpp_encoder = self.py_encoder if share_weights else AutoModel.from_pretrained(MODEL_NAME)
    def encode_py(self, tok):
        out = self.py_encoder(**tok).last_hidden_state
        return F.normalize(mean_pool(out, tok["attention_mask"]), dim=-1)
    def encode_cpp(self, tok):
        out = self.cpp_encoder(**tok).last_hidden_state
        return F.normalize(mean_pool(out, tok["attention_mask"]), dim=-1)

model = DualEncoder(share_weights=False).to(device)

loader = DataLoader(PairDataset(train), batch_size=BATCH_SIZE,
                    shuffle=True, collate_fn=collate, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
ce = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

model.train()
for epoch in range(EPOCHS):
    total = 0.0
    for step, (py_tok, cpp_tok) in enumerate(loader):
        py_tok = {k: v.to(device) for k, v in py_tok.items()}
        cpp_tok = {k: v.to(device) for k, v in cpp_tok.items()}
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            py_emb = model.encode_py(py_tok)
            cpp_emb = model.encode_cpp(cpp_tok)
            logits = py_emb @ cpp_emb.T / TEMPERATURE
            labels = torch.arange(logits.size(0), device=device)
            loss = (ce(logits, labels) + ce(logits.T, labels)) / 2
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
        if step % 50 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f}")
    print(f"epoch {epoch} avg loss {total/len(loader):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_6460/2845672122.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_6460/2845672122.py:80: Future

epoch 0 step 0 loss 0.9110
epoch 0 avg loss 0.7710
epoch 1 step 0 loss 0.2432
epoch 1 avg loss 0.1916
epoch 2 step 0 loss 0.1343
epoch 2 avg loss 0.0878
epoch 3 step 0 loss 0.0772
epoch 3 avg loss 0.0608


In [10]:
@torch.no_grad()
def encode_texts(texts, encode_fn, bs=32):
    model.eval()
    embs = []
    for i in range(0, len(texts), bs):
        tok = tokenizer(texts[i:i+bs], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
        tok = {k: v.to(device) for k, v in tok.items()}
        embs.append(encode_fn(tok).cpu())
    return torch.cat(embs).numpy()

query_ids   = test["datapointID"].tolist()[:200]          # adjust
py_queries  = test['source'].astype(str).tolist()[:200]     # adjust
cand_ids    = test["datapointID"].tolist()[200:]       # adjust
cpp_cands   = test['source'].astype(str).tolist()[200:]    # adjust

py_emb  = encode_texts(py_queries, model.encode_py)
cpp_emb = encode_texts(cpp_cands, model.encode_cpp)

sim = py_emb @ cpp_emb.T          # (n_queries, n_candidates)

TOP_K = 10                        # how many candidates per query; set to len(cand_ids) for full ranking

order = np.argsort(-sim, axis=1)  # indices of candidates, descending similarity

answers = []
for r in range(len(query_ids)):
    ranked = [cand_ids[c] for c in order[r, :TOP_K]]
    answers.append(";".join(ranked))

submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": query_ids,
    "answer": answers,
})
submission.to_csv("submission.csv", index=False)
print(submission.head())


   subtaskID          datapointID  \
0          1  py_test_public_0000   
1          1  py_test_public_0001   
2          1  py_test_public_0002   
3          1  py_test_public_0003   
4          1  py_test_public_0004   

                                              answer  
0  candidate_0055;candidate_0144;candidate_0006;c...  
1  candidate_0167;candidate_0058;candidate_0135;c...  
2  candidate_0007;candidate_0193;candidate_0107;c...  
3  candidate_0019;candidate_0114;candidate_0152;c...  
4  candidate_0101;candidate_0025;candidate_0007;c...  
